March Madness / college basketball prediction project centered on building a model that predicts NCAA tournament games from historical team-level data.

In [10]:
import bisect
from collections import defaultdict
import time
import warnings

import numpy as np
import optuna
import pandas as pd
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.base import clone
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [2]:
CURRENT_SEASON = 2026
DATA_DIR = 'march-machine-learning-mania-2026'
GENDERS = ['M', 'W']

# LR needs tournament upweighting because its single linear boundary is dominated by
# regular-season volume; tree models learn the tournament regime natively (weight=1 wins).
LR_TOURNAMENT_WEIGHT = 50

TOURNAMENT_DAY = 136  # approximate first round day
ORDINAL_RANK_FILL = 176  # neutral midpoint for teams with no ranking yet

# Ordinal ranking systems with perfect 24-season coverage (2003-2026)
# Used for computing cross-system stats (std dev, min rank)
ORDINAL_SYSTEMS = ['COL', 'DOL', 'MOR', 'POM', 'WLK']

# Subset exposed as individual features (maximizes methodological diversity)
# POM = adjusted efficiency, MOR = margin + SOS (highest deviation), COL = pure win/loss
ORDINAL_INDIVIDUAL_SYSTEMS = ['POM', 'MOR', 'COL']

# Allowlist of base features to include in training
# Raw counting stats excluded — their signal is captured by rates/efficiencies
ALLOWED_FEATURES = {
    # Shooting rates (pace-independent)
    'fg_pct', 'fg3_pct', 'ft_pct', 'fg3_rate',
    'opponent_fg_pct', 'opponent_fg3_pct', 'opponent_ft_pct', 'opponent_fg3_rate',
    # Dean Oliver's Four Factors
    'offensive_efg_pct', 'defensive_efg_pct',
    'offensive_tov_pct', 'defensive_tov_pct',
    'offensive_reb_pct', 'defensive_reb_pct',
    'offensive_ft_rate', 'defensive_ft_rate',
    # Efficiency ratings (Bayesian-updated, SOS-adjusted)
    'offensive_efficiency', 'defensive_efficiency',
    # Tempo
    'avg_possessions',
    # Defensive activity (own stats — more stable than opponent versions)
    'avg_stl', 'avg_blk',
    # Rebounds (raw — may remove in Tier 2)
    'avg_or', 'avg_dr', 'avg_opponent_or', 'avg_opponent_dr',
    # Turnovers (raw — may remove in Tier 2)
    'avg_to', 'avg_opponent_to',
    # Assists (raw — may remove in Tier 2)
    'avg_ast', 'avg_opponent_ast',
}

In [3]:
data = {os.path.splitext(f)[0]: pd.read_csv(os.path.join(DATA_DIR, f))
        for f in sorted(os.listdir(DATA_DIR)) if f.endswith('.csv')}

In [4]:
print(data[f'{GENDERS[0]}RegularSeasonDetailedResults'].columns)

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='str')


In [5]:
BOX_STATS = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
             'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']

PRIOR_OFFENSIVE_EFFICIENCY = 100.0
PRIOR_DEFENSIVE_EFFICIENCY = 100.0
EFFICIENCY_LEARNING_RATE = 0.10
LEAGUE_PRIOR_GAMES = 200
AVERAGE_POSSESSIONS_PER_TEAM = 69.0
HOME_COURT_ADVANTAGE = 3.5  # points per 100 possessions
SEASON_CARRYOVER_WEIGHT = 0.5
RESIDUAL_CAP = 15.0


def estimate_possessions(game, prefix):
    # FGA - OR + TO + 0.475 * FTA (standard Kenpom possession estimate)
    return game[f'{prefix}FGA'] - game[f'{prefix}OR'] + game[f'{prefix}TO'] + 0.475 * game[f'{prefix}FTA']


def update_team_box_stats(entry, game, own_prefix, opp_prefix):
    entry['game_count'] += 1
    for stat in BOX_STATS:
        entry['sums'][stat] += game[f'{own_prefix}{stat}']
        entry['sums'][f'opponent_{stat}'] += game[f'{opp_prefix}{stat}']


def extract_team_features(entry):
    """Extract features from a team's rolling entry."""
    game_count = entry['game_count']
    sums = entry['sums']
    features = {}

    # 28 per-game averages (14 own + 14 opponent)
    for stat in BOX_STATS:
        features[f'avg_{stat.lower()}'] = sums[stat] / game_count
        features[f'avg_opponent_{stat.lower()}'] = sums[f'opponent_{stat}'] / game_count

    # 8 rate stats (from sums for proper weighting)
    def safe_divide(numerator, denominator):
        return numerator / denominator if denominator > 0 else 0.0

    features['fg_pct'] = safe_divide(sums['FGM'], sums['FGA'])
    features['fg3_pct'] = safe_divide(sums['FGM3'], sums['FGA3'])
    features['ft_pct'] = safe_divide(sums['FTM'], sums['FTA'])
    features['fg3_rate'] = safe_divide(sums['FGA3'], sums['FGA'])
    features['opponent_fg_pct'] = safe_divide(sums['opponent_FGM'], sums['opponent_FGA'])
    features['opponent_fg3_pct'] = safe_divide(sums['opponent_FGM3'], sums['opponent_FGA3'])
    features['opponent_ft_pct'] = safe_divide(sums['opponent_FTM'], sums['opponent_FTA'])
    features['opponent_fg3_rate'] = safe_divide(sums['opponent_FGA3'], sums['opponent_FGA'])

    # Dean Oliver's Four Factors (offense and defense)
    # 1. Effective Field Goal % — eFG% = (FG + 0.5 * 3P) / FGA
    features['offensive_efg_pct'] = safe_divide(sums['FGM'] + 0.5 * sums['FGM3'], sums['FGA'])
    features['defensive_efg_pct'] = safe_divide(sums['opponent_FGM'] + 0.5 * sums['opponent_FGM3'], sums['opponent_FGA'])

    # 2. Turnover % — TOV% = TOV / (FGA + 0.44 * FTA + TOV)
    features['offensive_tov_pct'] = safe_divide(sums['TO'], sums['FGA'] + 0.44 * sums['FTA'] + sums['TO'])
    features['defensive_tov_pct'] = safe_divide(sums['opponent_TO'], sums['opponent_FGA'] + 0.44 * sums['opponent_FTA'] + sums['opponent_TO'])

    # 3. Rebound % — ORB% = ORB / (ORB + Opp DRB), DRB% = DRB / (Opp ORB + DRB)
    features['offensive_reb_pct'] = safe_divide(sums['OR'], sums['OR'] + sums['opponent_DR'])
    features['defensive_reb_pct'] = safe_divide(sums['DR'], sums['opponent_OR'] + sums['DR'])

    # 4. Free Throw Rate — FT / FGA
    features['offensive_ft_rate'] = safe_divide(sums['FTM'], sums['FGA'])
    features['defensive_ft_rate'] = safe_divide(sums['opponent_FTM'], sums['opponent_FGA'])

    # 3 efficiency
    features['offensive_efficiency'] = entry['offensive_efficiency']
    features['defensive_efficiency'] = entry['defensive_efficiency']
    features['net_efficiency'] = entry['offensive_efficiency'] - entry['defensive_efficiency']

    # 1 tempo
    features['avg_possessions'] = entry['total_possessions'] / game_count

    return features


def featurize_game(team_a_entry, team_b_entry, season, day_number, team_a_id, team_b_id,
                   label, team_a_location, is_tournament, team_a_seed, team_b_seed,
                   team_a_ordinals, team_b_ordinals):
    """Build a feature row from two teams' rolling entries (team_a = lower ID).

    Returns None if either team has no games yet.
    """
    if team_a_entry['game_count'] == 0 or team_b_entry['game_count'] == 0:
        return None

    team_a_features = extract_team_features(team_a_entry)
    team_b_features = extract_team_features(team_b_entry)

    row = {}
    for key, value in team_a_features.items():
        row[f'team_a_{key}'] = value
    for key, value in team_b_features.items():
        row[f'team_b_{key}'] = value
    for key in team_a_features:
        row[f'diff_{key}'] = team_a_features[key] - team_b_features[key]

    row['team_a_seed'] = team_a_seed
    row['team_b_seed'] = team_b_seed

    for key, value in team_a_ordinals.items():
        row[f'team_a_{key}'] = value
    for key, value in team_b_ordinals.items():
        row[f'team_b_{key}'] = value

    row['season'] = season
    row['day_number'] = day_number
    row['team_a_id'] = team_a_id
    row['team_b_id'] = team_b_id
    row['label'] = label
    row['team_a_location'] = team_a_location
    row['is_tournament'] = is_tournament

    return row


def build_pipeline(gender):
    """Build training data and prediction helper for one gender.

    Returns dict with training_data, feature_columns, feature_columns_no_seed,
    seed_lookup, predict_matchup, and has_ordinals.
    """
    has_ordinals = f'{gender}MasseyOrdinals' in data

    # Load gender-specific data
    regular_season_results = data[f'{gender}RegularSeasonDetailedResults'].copy()
    regular_season_results['is_tournament'] = 0
    tournament_results = data[f'{gender}NCAATourneyDetailedResults'].copy()
    tournament_results['is_tournament'] = 1
    results = pd.concat([regular_season_results, tournament_results], ignore_index=True).sort_values(
        ['Season', 'DayNum']).reset_index(drop=True)

    # Build tournament seed lookup: (season, team_id) -> seed number (1-16)
    seed_lookup = {}
    for _, seed_row in data[f'{gender}NCAATourneySeeds'].iterrows():
        seed_number = int(seed_row['Seed'][1:3])  # "W01" -> 1, "X16a" -> 16
        seed_lookup[(seed_row['Season'], seed_row['TeamID'])] = seed_number

    # Build ordinal lookups (only when ordinal data exists)
    if has_ordinals:
        ordinal_averages = (data[f'{gender}MasseyOrdinals']
            .groupby(['Season', 'RankingDayNum', 'TeamID'])['OrdinalRank']
            .mean()
            .reset_index())

        ordinal_lookup = defaultdict(list)
        for _, row in ordinal_averages.iterrows():
            ordinal_lookup[(row['Season'], row['TeamID'])].append(
                (row['RankingDayNum'], row['OrdinalRank']))
        for key in ordinal_lookup:
            ordinal_lookup[key].sort()

        selected_ordinals = data[f'{gender}MasseyOrdinals'][
            data[f'{gender}MasseyOrdinals']['SystemName'].isin(ORDINAL_SYSTEMS)]

        system_ordinal_lookup = defaultdict(list)
        for _, row in selected_ordinals.iterrows():
            system_ordinal_lookup[(row['SystemName'], row['Season'], row['TeamID'])].append(
                (row['RankingDayNum'], row['OrdinalRank']))
        for key in system_ordinal_lookup:
            system_ordinal_lookup[key].sort()

        ordinal_stats_grouped = (selected_ordinals
            .groupby(['Season', 'RankingDayNum', 'TeamID'])['OrdinalRank']
            .agg(['std', 'min'])
            .reset_index())

        ordinal_stats_lookup = defaultdict(list)
        for _, row in ordinal_stats_grouped.iterrows():
            std_value = row['std'] if not np.isnan(row['std']) else 0.0
            ordinal_stats_lookup[(row['Season'], row['TeamID'])].append(
                (row['RankingDayNum'], std_value, row['min']))
        for key in ordinal_stats_lookup:
            ordinal_stats_lookup[key].sort()

        print(f"[{gender}] Ordinal lookup: {len(ordinal_lookup)}, system: {len(system_ordinal_lookup)}, stats: {len(ordinal_stats_lookup)}")
    else:
        ordinal_lookup = {}
        system_ordinal_lookup = {}
        ordinal_stats_lookup = {}
        print(f"[{gender}] No ordinal data available")

    # --- Ordinal helper closures ---

    def get_ordinal_rank(season, team_id, day_num):
        rankings = ordinal_lookup.get((season, team_id))
        if not rankings:
            return np.nan
        day_nums = [r[0] for r in rankings]
        idx = bisect.bisect_right(day_nums, day_num) - 1
        if idx < 0:
            return np.nan
        return rankings[idx][1]

    def get_system_ordinal_rank(system_name, season, team_id, day_num):
        rankings = system_ordinal_lookup.get((system_name, season, team_id))
        if not rankings:
            return np.nan
        day_nums = [r[0] for r in rankings]
        idx = bisect.bisect_right(day_nums, day_num) - 1
        if idx < 0:
            return np.nan
        return rankings[idx][1]

    def get_ordinal_stats(season, team_id, day_num):
        stats = ordinal_stats_lookup.get((season, team_id))
        if not stats:
            return np.nan, np.nan
        day_nums = [r[0] for r in stats]
        idx = bisect.bisect_right(day_nums, day_num) - 1
        if idx < 0:
            return np.nan, np.nan
        return stats[idx][1], stats[idx][2]

    def build_ordinal_features(season, team_id, day_num):
        if not has_ordinals:
            return {}
        features = {}
        features['ordinal_rank'] = get_ordinal_rank(season, team_id, day_num)
        for system_name in ORDINAL_INDIVIDUAL_SYSTEMS:
            features[f'{system_name.lower()}_rank'] = get_system_ordinal_rank(system_name, season, team_id, day_num)
        std_dev, min_rank = get_ordinal_stats(season, team_id, day_num)
        features['ordinal_std'] = std_dev
        features['ordinal_min'] = min_rank
        return features

    # --- Rolling stats closures ---

    team_rolling = {}
    league_totals = {}

    def get_or_create_entry(season, team_id):
        key = (season, team_id)
        if key not in team_rolling:
            sums = {}
            for stat in BOX_STATS:
                sums[stat] = 0.0
                sums[f'opponent_{stat}'] = 0.0

            prior_key = (season - 1, team_id)
            if prior_key in team_rolling:
                prior = team_rolling[prior_key]
                initial_offensive = (SEASON_CARRYOVER_WEIGHT * prior['offensive_efficiency']
                                     + (1 - SEASON_CARRYOVER_WEIGHT) * PRIOR_OFFENSIVE_EFFICIENCY)
                initial_defensive = (SEASON_CARRYOVER_WEIGHT * prior['defensive_efficiency']
                                     + (1 - SEASON_CARRYOVER_WEIGHT) * PRIOR_DEFENSIVE_EFFICIENCY)
            else:
                initial_offensive = PRIOR_OFFENSIVE_EFFICIENCY
                initial_defensive = PRIOR_DEFENSIVE_EFFICIENCY

            team_rolling[key] = {
                'sums': sums,
                'game_count': 0,
                'offensive_efficiency': initial_offensive,
                'defensive_efficiency': initial_defensive,
                'total_possessions': 0.0,
            }
        return team_rolling[key]

    def get_or_create_league_totals(season):
        if season not in league_totals:
            league_totals[season] = {'total_points': 0.0, 'total_possessions': 0.0, 'game_count': 0}
        return league_totals[season]

    def get_league_average_efficiency(season):
        totals = get_or_create_league_totals(season)
        phantom_games = max(0, LEAGUE_PRIOR_GAMES - totals['game_count'])
        if phantom_games == 0:
            return totals['total_points'] / totals['total_possessions'] * 100.0
        if season - 1 in league_totals and league_totals[season - 1]['total_possessions'] > 0:
            prior_efficiency = (league_totals[season - 1]['total_points']
                                / league_totals[season - 1]['total_possessions'] * 100.0)
        else:
            prior_efficiency = 100.0
        phantom_possessions = phantom_games * 2 * AVERAGE_POSSESSIONS_PER_TEAM
        phantom_points = phantom_possessions * prior_efficiency / 100.0
        blended_points = phantom_points + totals['total_points']
        blended_possessions = phantom_possessions + totals['total_possessions']
        return blended_points / blended_possessions * 100.0

    # --- Main game-processing loop ---

    training_rows = []

    for _, game in results.iterrows():
        winner_entry = get_or_create_entry(game['Season'], game['WTeamID'])
        loser_entry = get_or_create_entry(game['Season'], game['LTeamID'])

        winner_team_id = game['WTeamID']
        loser_team_id = game['LTeamID']

        if game['is_tournament'] == 1:
            winner_seed = seed_lookup.get((game['Season'], winner_team_id), np.nan)
            loser_seed = seed_lookup.get((game['Season'], loser_team_id), np.nan)
        else:
            winner_seed = np.nan
            loser_seed = np.nan

        winner_ordinals = build_ordinal_features(game['Season'], winner_team_id, game['DayNum'])
        loser_ordinals = build_ordinal_features(game['Season'], loser_team_id, game['DayNum'])

        if winner_team_id < loser_team_id:
            team_a_location = game['WLoc']
            row = featurize_game(winner_entry, loser_entry, game['Season'], game['DayNum'],
                                 winner_team_id, loser_team_id, label=1, team_a_location=team_a_location,
                                 is_tournament=game['is_tournament'],
                                 team_a_seed=winner_seed, team_b_seed=loser_seed,
                                 team_a_ordinals=winner_ordinals, team_b_ordinals=loser_ordinals)
        else:
            location_flip = {'H': 'A', 'A': 'H', 'N': 'N'}
            team_a_location = location_flip[game['WLoc']]
            row = featurize_game(loser_entry, winner_entry, game['Season'], game['DayNum'],
                                 loser_team_id, winner_team_id, label=0, team_a_location=team_a_location,
                                 is_tournament=game['is_tournament'],
                                 team_a_seed=loser_seed, team_b_seed=winner_seed,
                                 team_a_ordinals=loser_ordinals, team_b_ordinals=winner_ordinals)
        if row is not None:
            training_rows.append(row)

        # --- Adjusted efficiency update ---
        possessions = (estimate_possessions(game, 'W') + estimate_possessions(game, 'L')) / 2

        if possessions > 0:
            winner_raw_offensive = game['WScore'] / possessions * 100.0
            loser_raw_offensive = game['LScore'] / possessions * 100.0

            if game['WLoc'] == 'H':
                winner_raw_offensive -= HOME_COURT_ADVANTAGE
                loser_raw_offensive += HOME_COURT_ADVANTAGE
            elif game['WLoc'] == 'A':
                winner_raw_offensive += HOME_COURT_ADVANTAGE
                loser_raw_offensive -= HOME_COURT_ADVANTAGE

            league_average = get_league_average_efficiency(game['Season'])

            winner_expected_offensive = (winner_entry['offensive_efficiency']
                                         + loser_entry['defensive_efficiency']
                                         - league_average)
            loser_expected_offensive = (loser_entry['offensive_efficiency']
                                        + winner_entry['defensive_efficiency']
                                        - league_average)

            winner_offensive_residual = np.clip(winner_raw_offensive - winner_expected_offensive, -RESIDUAL_CAP, RESIDUAL_CAP)
            loser_offensive_residual = np.clip(loser_raw_offensive - loser_expected_offensive, -RESIDUAL_CAP, RESIDUAL_CAP)

            winner_entry['offensive_efficiency'] += EFFICIENCY_LEARNING_RATE * winner_offensive_residual
            loser_entry['defensive_efficiency'] += EFFICIENCY_LEARNING_RATE * winner_offensive_residual
            loser_entry['offensive_efficiency'] += EFFICIENCY_LEARNING_RATE * loser_offensive_residual
            winner_entry['defensive_efficiency'] += EFFICIENCY_LEARNING_RATE * loser_offensive_residual

        season_totals = get_or_create_league_totals(game['Season'])
        season_totals['total_points'] += game['WScore'] + game['LScore']
        season_totals['total_possessions'] += 2 * possessions
        season_totals['game_count'] += 1

        winner_entry['total_possessions'] += possessions
        loser_entry['total_possessions'] += possessions

        update_team_box_stats(winner_entry, game, 'W', 'L')
        update_team_box_stats(loser_entry, game, 'L', 'W')

    print(f"[{gender}] Processed {len(results)} games, {len(team_rolling)} team-seasons")
    print(f"[{gender}] Training rows: {len(training_rows)}, seed lookup: {len(seed_lookup)}")

    # --- Build training DataFrame ---

    training_data = pd.DataFrame(training_rows)

    # --- Feature selection ---

    ordinal_fill_columns = {'rank': [], 'std': [], 'min': []}

    if has_ordinals:
        ordinal_rank_columns = [col for col in training_data.columns
                                if col.endswith('_rank') and ('ordinal' in col or 'pom' in col or 'mor' in col or 'col' in col)]
        ordinal_std_columns = [col for col in training_data.columns if col.endswith('_ordinal_std')]
        ordinal_min_columns = [col for col in training_data.columns if col.endswith('_ordinal_min')]

        for col in ordinal_rank_columns:
            training_data[col] = training_data[col].fillna(ORDINAL_RANK_FILL)
        for col in ordinal_std_columns:
            training_data[col] = training_data[col].fillna(0.0)
        for col in ordinal_min_columns:
            training_data[col] = training_data[col].fillna(ORDINAL_RANK_FILL)

        ordinal_fill_columns = {'rank': ordinal_rank_columns, 'std': ordinal_std_columns, 'min': ordinal_min_columns}

        print(f"[{gender}] Ordinal rank coverage: {(training_data['team_a_ordinal_rank'] != ORDINAL_RANK_FILL).mean():.1%}")

    allowed_columns = set()
    for base_name in ALLOWED_FEATURES:
        allowed_columns.add(f'team_a_{base_name}')
        allowed_columns.add(f'team_b_{base_name}')
    if has_ordinals:
        ordinal_feature_names = {'ordinal_rank', 'ordinal_std', 'ordinal_min'}
        for sys in ORDINAL_INDIVIDUAL_SYSTEMS:
            ordinal_feature_names.add(f'{sys.lower()}_rank')
        for base_name in ordinal_feature_names:
            allowed_columns.add(f'team_a_{base_name}')
            allowed_columns.add(f'team_b_{base_name}')
    allowed_columns.add('team_a_seed')
    allowed_columns.add('team_b_seed')

    metadata_columns = {"season", "day_number", "team_a_id", "team_b_id", "label",
                        "team_a_location", "is_tournament"}
    feature_columns = [col for col in training_data.columns
                       if col in allowed_columns and col not in metadata_columns]

    seed_columns = {"team_a_seed", "team_b_seed"}
    feature_columns_no_seed = [col for col in feature_columns if col not in seed_columns]

    print(f"[{gender}] {len(feature_columns)} features (incl seed), {len(feature_columns_no_seed)} without seed")

    # --- Prediction closure ---

    def predict_matchup(model, team_id_1, team_id_2, seed_num_1, seed_num_2):
        """Predict win probability for team_id_1 using bidirectional averaging."""
        entry_1 = team_rolling[(CURRENT_SEASON, team_id_1)]
        entry_2 = team_rolling[(CURRENT_SEASON, team_id_2)]
        ordinals_1 = build_ordinal_features(CURRENT_SEASON, team_id_1, TOURNAMENT_DAY)
        ordinals_2 = build_ordinal_features(CURRENT_SEASON, team_id_2, TOURNAMENT_DAY)

        lower_id = min(team_id_1, team_id_2)
        higher_id = max(team_id_1, team_id_2)

        if team_id_1 < team_id_2:
            lower_entry, higher_entry = entry_1, entry_2
            lower_seed, higher_seed = seed_num_1, seed_num_2
            lower_ordinals, higher_ordinals = ordinals_1, ordinals_2
        else:
            lower_entry, higher_entry = entry_2, entry_1
            lower_seed, higher_seed = seed_num_2, seed_num_1
            lower_ordinals, higher_ordinals = ordinals_2, ordinals_1

        row_forward = featurize_game(
            lower_entry, higher_entry, CURRENT_SEASON, TOURNAMENT_DAY,
            lower_id, higher_id, label=0, team_a_location='N',
            is_tournament=1, team_a_seed=lower_seed, team_b_seed=higher_seed,
            team_a_ordinals=lower_ordinals, team_b_ordinals=higher_ordinals)

        row_reversed = featurize_game(
            higher_entry, lower_entry, CURRENT_SEASON, TOURNAMENT_DAY,
            higher_id, lower_id, label=0, team_a_location='N',
            is_tournament=1, team_a_seed=higher_seed, team_b_seed=lower_seed,
            team_a_ordinals=higher_ordinals, team_b_ordinals=lower_ordinals)

        prediction_rows = pd.DataFrame([row_forward, row_reversed])
        for col in ordinal_fill_columns['rank']:
            prediction_rows[col] = prediction_rows[col].fillna(ORDINAL_RANK_FILL)
        for col in ordinal_fill_columns['std']:
            prediction_rows[col] = prediction_rows[col].fillna(0.0)
        for col in ordinal_fill_columns['min']:
            prediction_rows[col] = prediction_rows[col].fillna(ORDINAL_RANK_FILL)

        X_forward = prediction_rows.iloc[[0]][feature_columns].values
        X_reversed = prediction_rows.iloc[[1]][feature_columns].values

        prob_lower_wins_forward = model.predict_proba(X_forward)[0, 1]
        prob_higher_wins_reversed = model.predict_proba(X_reversed)[0, 1]

        prob_lower_wins = (prob_lower_wins_forward + (1 - prob_higher_wins_reversed)) / 2

        if team_id_1 == lower_id:
            return prob_lower_wins
        else:
            return 1 - prob_lower_wins

    return {
        'training_data': training_data,
        'feature_columns': feature_columns,
        'feature_columns_no_seed': feature_columns_no_seed,
        'seed_lookup': seed_lookup,
        'predict_matchup': predict_matchup,
        'has_ordinals': has_ordinals,
    }


pipelines = {gender: build_pipeline(gender) for gender in GENDERS}

[M] Ordinal lookup: 8356, system: 41729, stats: 8355
[M] Processed 125978 games, 8346 team-seasons
[M] Training rows: 120861, seed lookup: 2694
[M] Ordinal rank coverage: 97.3%
[M] 72 features (incl seed), 70 without seed
[W] No ordinal data available
[W] Processed 88148 games, 5965 team-seasons
[W] Training rows: 84535, seed lookup: 1812
[W] 60 features (incl seed), 58 without seed


### Cross-validation results (model selection)

LightGBM was selected based on these results. The code is commented out below.

```
=== M: Cross-validation (10 folds, seasons 2016–2025) ===

Model                  Feats            ROC AUC              Brier        Tourney AUC      Tourney Brier
======================================================================================================
LogisticRegression        70 0.7791 ± 0.0083   0.1920 ± 0.0038   0.7698 ± 0.0553   0.1925 ± 0.0204
HistGradientBoosting      72 0.7806 ± 0.0083   0.1903 ± 0.0035   0.7775 ± 0.0541   0.1911 ± 0.0204
XGBoost                   72 0.7806 ± 0.0088   0.1903 ± 0.0037   0.7742 ± 0.0527   0.1913 ± 0.0205
LightGBM                  72 0.7803 ± 0.0083   0.1905 ± 0.0035   0.7730 ± 0.0537   0.1908 ± 0.0218

=== W: Cross-validation (10 folds, seasons 2016–2025) ===

Model                  Feats            ROC AUC              Brier        Tourney AUC      Tourney Brier
======================================================================================================
LogisticRegression        58 0.8284 ± 0.0100   0.1712 ± 0.0089   0.8636 ± 0.0343   0.1606 ± 0.0268
HistGradientBoosting      60 0.8309 ± 0.0089   0.1678 ± 0.0044   0.8662 ± 0.0386   0.1552 ± 0.0241
XGBoost                   60 0.8308 ± 0.0090   0.1679 ± 0.0045   0.8684 ± 0.0361   0.1543 ± 0.0241
LightGBM                  60 0.8306 ± 0.0091   0.1680 ± 0.0045   0.8627 ± 0.0335   0.1576 ± 0.0243
```

In [ ]:
# # --- Cross-validation ---
#
# VALIDATION_SEASONS = list(range(2016, 2026))
#
# models = {
#     "LogisticRegression": LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs"),
#     "HistGradientBoosting": HistGradientBoostingClassifier(
#         max_iter=200, max_depth=4, learning_rate=0.1, random_state=42,
#     ),
#     "XGBoost": XGBClassifier(
#         n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42,
#         eval_metric="logloss",
#     ),
#     "LightGBM": LGBMClassifier(
#         n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42,
#         verbose=-1,
#     ),
# }
#
# for gender in GENDERS:
#     pipeline = pipelines[gender]
#     training_data = pipeline['training_data']
#     feature_columns = pipeline['feature_columns']
#     feature_columns_no_seed = pipeline['feature_columns_no_seed']
#
#     y = training_data["label"].values
#     seasons = training_data["season"].values
#     is_tournament = training_data["is_tournament"].values
#     lr_sample_weights = np.where(is_tournament == 1, LR_TOURNAMENT_WEIGHT, 1.0)
#
#     folds = []
#     for holdout_season in VALIDATION_SEASONS:
#         train_mask = seasons < holdout_season
#         validation_mask = seasons == holdout_season
#         if train_mask.sum() > 0 and validation_mask.sum() > 0:
#             folds.append((np.where(train_mask)[0], np.where(validation_mask)[0]))
#
#     results_table = []
#
#     for model_name, model_template in models.items():
#         fold_aucs, fold_briers = [], []
#         tournament_aucs, tournament_briers = [], []
#
#         if model_name == "LogisticRegression":
#             model_features = feature_columns_no_seed
#             use_scaler = True
#         else:
#             model_features = feature_columns
#             use_scaler = False
#
#         X_model = training_data[model_features].values
#
#         for fold_index, (train_indices, validation_indices) in enumerate(folds):
#             if use_scaler:
#                 scaler = StandardScaler()
#                 X_train = scaler.fit_transform(X_model[train_indices])
#                 X_validation = scaler.transform(X_model[validation_indices])
#             else:
#                 X_train = X_model[train_indices]
#                 X_validation = X_model[validation_indices]
#
#             y_train = y[train_indices]
#             y_validation = y[validation_indices]
#
#             model = clone(model_template)
#             fit_kwargs = {}
#             if model_name == "LogisticRegression":
#                 fit_kwargs["sample_weight"] = lr_sample_weights[train_indices]
#             model.fit(X_train, y_train, **fit_kwargs)
#             probabilities = model.predict_proba(X_validation)[:, 1]
#
#             fold_aucs.append(roc_auc_score(y_validation, probabilities))
#             fold_briers.append(brier_score_loss(y_validation, probabilities))
#
#             tournament_mask = is_tournament[validation_indices] == 1
#             if tournament_mask.sum() > 0:
#                 tournament_aucs.append(roc_auc_score(y_validation[tournament_mask], probabilities[tournament_mask]))
#                 tournament_briers.append(brier_score_loss(y_validation[tournament_mask], probabilities[tournament_mask]))
#
#         results_table.append({
#             "model": model_name,
#             "features": len(model_features),
#             "auc_mean": np.mean(fold_aucs),
#             "auc_std": np.std(fold_aucs),
#             "brier_mean": np.mean(fold_briers),
#             "brier_std": np.std(fold_briers),
#             "tournament_auc_mean": np.mean(tournament_aucs),
#             "tournament_auc_std": np.std(tournament_aucs),
#             "tournament_brier_mean": np.mean(tournament_briers),
#             "tournament_brier_std": np.std(tournament_briers),
#         })
#
#     print(f"\n=== {gender}: Cross-validation ({len(folds)} folds, seasons {VALIDATION_SEASONS[0]}–{VALIDATION_SEASONS[-1]}) ===\n")
#     print(f"{'Model':<22} {'Feats':>5} {'ROC AUC':>18} {'Brier':>18} {'Tourney AUC':>18} {'Tourney Brier':>18}")
#     print("=" * 102)
#     for row in results_table:
#         print(f"{row['model']:<22} "
#               f"{row['features']:>5} "
#               f"{row['auc_mean']:.4f} ± {row['auc_std']:.4f}   "
#               f"{row['brier_mean']:.4f} ± {row['brier_std']:.4f}   "
#               f"{row['tournament_auc_mean']:.4f} ± {row['tournament_auc_std']:.4f}   "
#               f"{row['tournament_brier_mean']:.4f} ± {row['tournament_brier_std']:.4f}")

In [ ]:
# --- Optuna hyperparameter tuning for LightGBM ---
NUMBER_OF_TUNING_TRIALS = 10
EARLY_STOPPING_ROUNDS = 50
N_ESTIMATORS_CEILING = 2000

optuna.logging.set_verbosity(optuna.logging.WARNING)

best_params = {}

for gender in GENDERS:
    pipeline = pipelines[gender]
    training_data = pipeline['training_data']
    feature_columns = pipeline['feature_columns']

    y = training_data["label"].values
    seasons = training_data["season"].values
    is_tournament = training_data["is_tournament"].values

    X_tuning = training_data[feature_columns].values

    folds = []
    for holdout_season in VALIDATION_SEASONS:
        train_mask = seasons < holdout_season
        validation_mask = seasons == holdout_season
        if train_mask.sum() > 0 and validation_mask.sum() > 0:
            folds.append((np.where(train_mask)[0], np.where(validation_mask)[0]))

    def objective(trial):
        params = {
            "n_estimators": N_ESTIMATORS_CEILING,
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "random_state": 42,
            "verbose": -1,
        }

        tournament_briers = []
        for fold_index, (train_indices, validation_indices) in enumerate(folds):
            model = LGBMClassifier(**params)
            model.fit(
                X_tuning[train_indices], y[train_indices],
                eval_set=[(X_tuning[validation_indices], y[validation_indices])],
                callbacks=[
                    early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                    log_evaluation(period=0),
                ],
            )
            probabilities = model.predict_proba(X_tuning[validation_indices])[:, 1]

            tournament_mask = is_tournament[validation_indices] == 1
            if tournament_mask.sum() > 0:
                tournament_briers.append(
                    brier_score_loss(y[validation_indices][tournament_mask], probabilities[tournament_mask])
                )

            if len(tournament_briers) > 0:
                trial.report(np.mean(tournament_briers), fold_index)
                if trial.should_prune():
                    raise optuna.TrialPruned()

        return np.mean(tournament_briers)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3),
    )

    start_time = time.time()
    study.optimize(objective, n_trials=NUMBER_OF_TUNING_TRIALS)
    elapsed_time = time.time() - start_time

    gender_best = study.best_params
    gender_best["random_state"] = 42
    gender_best["verbose"] = -1
    best_params[gender] = gender_best

    pruned_count = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])

    print(f"\n=== {gender} ===")
    print(f"Tuning: {NUMBER_OF_TUNING_TRIALS} trials in {elapsed_time:.1f}s ({elapsed_time / NUMBER_OF_TUNING_TRIALS:.1f}s/trial)")
    print(f"  Pruned: {pruned_count}/{NUMBER_OF_TUNING_TRIALS}")
    print(f"Best tournament Brier: {study.best_value:.4f}")
    print(f"Best parameters:")
    for key, value in gender_best.items():
        print(f"  {key}: {value}")

In [ ]:
# --- Train final models (LightGBM on all data with tuned parameters) ---
final_models = {}

for gender in GENDERS:
    pipeline = pipelines[gender]
    training_data = pipeline['training_data']
    feature_columns = pipeline['feature_columns']

    X_all = training_data[feature_columns].values
    y_all = training_data['label'].values
    seasons_all = training_data['season'].values

    final_train_mask = seasons_all < CURRENT_SEASON - 1
    final_eval_mask = seasons_all == CURRENT_SEASON - 1

    final_params = best_params[gender].copy()
    final_params["n_estimators"] = N_ESTIMATORS_CEILING

    model = LGBMClassifier(**final_params)
    model.fit(
        X_all[final_train_mask], y_all[final_train_mask],
        eval_set=[(X_all[final_eval_mask], y_all[final_eval_mask])],
        callbacks=[
            early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
            log_evaluation(period=0),
        ],
    )
    optimal_trees = model.best_iteration_

    model = LGBMClassifier(**{**final_params, "n_estimators": optimal_trees})
    model.fit(X_all, y_all)
    final_models[gender] = model

    print(f"[{gender}] LightGBM: {optimal_trees} trees (early stopped), trained on {len(y_all)} games")

In [9]:
# --- 2026 First Round Predictions (Sanity Check) ---

region_labels = {'W': 'WEST', 'X': 'EAST', 'Y': 'SOUTH', 'Z': 'MIDWEST'}
gender_labels = {'M': "MEN'S", 'W': "WOMEN'S"}

for gender in GENDERS:
    pipeline = pipelines[gender]
    seed_lookup = pipeline['seed_lookup']
    predict_matchup = pipeline['predict_matchup']
    model = final_models[gender]

    # Build lookups for current season
    seed_to_team = {}
    for _, row in data[f'{gender}NCAATourneySeeds'][data[f'{gender}NCAATourneySeeds']['Season'] == CURRENT_SEASON].iterrows():
        seed_to_team[row['Seed']] = row['TeamID']

    team_names = dict(zip(data[f'{gender}Teams']['TeamID'], data[f'{gender}Teams']['TeamName']))

    # Resolve play-in games
    slots_current = data[f'{gender}NCAATourneySlots'][data[f'{gender}NCAATourneySlots']['Season'] == CURRENT_SEASON]
    playin_slots = slots_current[~slots_current['Slot'].str.startswith('R')]
    first_round_slots = slots_current[slots_current['Slot'].str.startswith('R1')]

    print(f"\n{'=' * 60}")
    print(f"  {gender_labels[gender]} TOURNAMENT")
    print(f"{'=' * 60}")

    if len(playin_slots) > 0:
        print("\n=== PLAY-IN GAMES ===\n")
        playin_winners = {}
        for _, slot in playin_slots.iterrows():
            strong_team = seed_to_team[slot['StrongSeed']]
            weak_team = seed_to_team[slot['WeakSeed']]
            strong_seed_num = seed_lookup[(CURRENT_SEASON, strong_team)]
            weak_seed_num = seed_lookup[(CURRENT_SEASON, weak_team)]

            prob_strong_wins = predict_matchup(model, strong_team, weak_team, strong_seed_num, weak_seed_num)
            winner_id = strong_team if prob_strong_wins > 0.5 else weak_team
            playin_winners[slot['Slot']] = winner_id

            marker = "←" if prob_strong_wins > 0.5 else " "
            print(f"  {slot['Slot']:4s}: ({strong_seed_num:>2d}) {team_names[strong_team]:<22s} {prob_strong_wins:5.1%}  {marker}")
            marker = " " if prob_strong_wins > 0.5 else "←"
            print(f"        ({weak_seed_num:>2d}) {team_names[weak_team]:<22s} {1 - prob_strong_wins:5.1%}  {marker}")
            print()
    else:
        playin_winners = {}

    # Predict all first round games
    predictions = []

    for _, slot in first_round_slots.iterrows():
        region = slot['Slot'][2]  # R1W1 -> 'W'

        strong_seed_str = slot['StrongSeed']
        strong_team = playin_winners.get(strong_seed_str, seed_to_team.get(strong_seed_str))

        weak_seed_str = slot['WeakSeed']
        weak_team = playin_winners.get(weak_seed_str, seed_to_team.get(weak_seed_str))

        strong_seed_num = seed_lookup[(CURRENT_SEASON, strong_team)]
        weak_seed_num = seed_lookup[(CURRENT_SEASON, weak_team)]

        prob_strong_wins = predict_matchup(model, strong_team, weak_team, strong_seed_num, weak_seed_num)

        predictions.append({
            'region': region,
            'strong_seed': strong_seed_num,
            'weak_seed': weak_seed_num,
            'strong_team': team_names[strong_team],
            'weak_team': team_names[weak_team],
            'strong_seed_win_prob': prob_strong_wins,
        })

    print("\n=== FIRST ROUND PREDICTIONS ===\n")
    for region_code in ['W', 'X', 'Y', 'Z']:
        region_preds = [p for p in predictions if p['region'] == region_code]
        if not region_preds:
            continue
        region_preds.sort(key=lambda p: p['strong_seed'])
        print(f"--- {region_labels[region_code]} ---")
        print(f"  {'Matchup':<52s} {'Fav %':>5s}")
        print(f"  {'─' * 58}")
        for p in region_preds:
            matchup = f"({p['strong_seed']:>2d}) {p['strong_team']:<22s} vs ({p['weak_seed']:>2d}) {p['weak_team']}"
            print(f"  {matchup:<52s} {p['strong_seed_win_prob']:5.1%}")
        print()


  MEN'S TOURNAMENT

=== PLAY-IN GAMES ===

  X16 : (16) Lehigh                 53.0%  ←
        (16) Prairie View           47.0%   

  Y11 : (11) Miami OH               39.0%   
        (11) SMU                    61.0%  ←

  Y16 : (16) Howard                 48.4%   
        (16) UMBC                   51.6%  ←

  Z11 : (11) NC State               48.3%   
        (11) Texas                  51.7%  ←


=== FIRST ROUND PREDICTIONS ===

--- WEST ---
  Matchup                                              Fav %
  ──────────────────────────────────────────────────────────
  ( 1) Duke                   vs (16) Siena            93.7%
  ( 2) Connecticut            vs (15) Furman           92.7%
  ( 3) Michigan St            vs (14) N Dakota St      88.8%
  ( 4) Kansas                 vs (13) Cal Baptist      77.4%
  ( 5) St John's              vs (12) Northern Iowa    78.3%
  ( 6) Louisville             vs (11) South Florida    60.5%
  ( 7) UCLA                   vs (10) UCF              56